In [22]:
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import contractions

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [25]:
def remove_stopwords(text):
    SENTIMENT_CRITICAL_WORDS = {
        # Negations
        'not', 'no', 'never',
        'neither', 'nor', 'none', 'nobody',
        'nothing', 'nowhere', 'hardly', 'scarcely', 'barely',

        # Intensifiers
        'very', 'really', 'absolutely', 'completely', 'extremely',
        'quite', 'rather', 'pretty', 'too', 'enough',

        # Negation indicators
        'but', 'however', 'although', 'though', 'yet', 'because',

        # Modal verbs (express certainty/doubt)
        'should', 'would', 'wouldn', "wouldn't",
        'could', 'might', 'mightn', "mightn't", 'may', 'must',
        "mustn't", 'mustn',
        'can', 'cannot', "can't", 'will', "won't", 'won',
        'couldn', "couldn't", 'don', "don't",
        'should', "should've", 'shouldn', "shouldn't",
    }
    STOPWORDS = set(stopwords.words('english'))

    tokens = word_tokenize(text)
    filtered_stopwords_subset = STOPWORDS - SENTIMENT_CRITICAL_WORDS
    filtered_tokens = [word for word in tokens if word not in filtered_stopwords_subset]

    return ' '.join(filtered_tokens)


def stemming(text):
    tokens = word_tokenize(text)
    stemmer = SnowballStemmer("english")
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    return ' '.join(stemmed_tokens)


def handle_emotes(text):
    return re.sub(r'(:\s?\)|:-\s?\)|:\s?D|:-\s?D|:\s?\(|:-\s?\(|;-\s?\)|;\s?\)|:P|:-P|:-O|:O|<3)', '', text)


def remove_noisy_punctuations(text):
    return re.sub(r'[\"#$%&\'()*+\-\/:;<=>@\[\\\]^_`{|}~]', '', text)


def remove_redundant_punctuation(text):
    return re.sub(r'([!\"#$%&\'()*+,\-.\/:;<=>?@\[\\\]^_`{|}~ ])\1+', r'\1', text)


def preprocess_text(text, use_stemming=False, include_stopwords=False):
    text = text.lower()

    # Strip Urls
    text = re.sub(r'http\S+', '', text)

    # Strip HTML Tags
    text = re.sub(r'<.*?>', '', text)

    # Expand Contractions
    text = contractions.fix(text)

    # Remove Numbers
    text = re.sub(r'\d+', '', text)

    # Note(Tony): TextVectorization as can remove punctuations.
    text = handle_emotes(text)
    text = remove_noisy_punctuations(text)
    text = remove_redundant_punctuation(text)

    if not include_stopwords:
        text = remove_stopwords(text)

    if use_stemming:
        text = stemming(text)

    # Remove extra whitespaces
    text = ' '.join(text.split())

    return text

In [26]:
MODEL_FILE_PATH = '../models/tensorflow_model.keras'

model = tf.keras.models.load_model(MODEL_FILE_PATH)
print(f"Loaded Model: {model.name}")

Loaded Model: single_layer_model


In [27]:
labels = ['Bad', 'Excellent', 'Good', 'Very bad', 'Very good']
int_to_label = {0: 'Bad', 1: 'Excellent', 2: 'Good', 3: 'Very bad', 4: 'Very good'}
label_to_int = {v: k for k, v in int_to_label.items()}

In [32]:
TEST_FILE = '../data/Test cases.xlsx'

df_test = (pd.read_excel(TEST_FILE))
df_test['text'] = df_test['text'].apply(lambda x: preprocess_text(x))

X = np.array(df_test['text'])
y_encoded = df_test['text'].apply(lambda x: label_to_int[x])

In [33]:
y_pred_probs = model.predict(X)
y_pred       = np.argmax(y_pred_probs, axis=1)
test_accuracy = accuracy_score(y_encoded, y_pred)
print(f"Test accuracy: {test_accuracy:.4f}")

cm   = confusion_matrix(y_encoded, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Reds, colorbar=False)

plt.show()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step


ValueError: Found input variables with inconsistent numbers of samples: [7000, 6]